# Model XGBoost untuk prediksi limbah makanan

**Tujuan notebook:** menyelesaikan dua eksperimen XGBoost dengan Standard dan Robust secara terbuka, langkah demi langkah.

XGBoost menambah pohon secara bertahap untuk memperbaiki kesalahan model sebelumnya. Dua kandidat memakai kedalaman 3/5 dengan regularisasi 5/10. Learning rate 0,04, maksimal 500 tahap, dan early stopping 40 tahap. Gradient boosting memakai gradien loss untuk menyusun pohon baru; prosesnya berbeda dari pembaruan bobot jaringan saraf.

**Pertanyaan yang dijawab:** Seberapa besar kesalahan dalam kg? Apakah Robust membantu? Apakah training terlalu bagus dibanding validasi? Fitur apa yang membantu model? Seberapa cepat prediksinya? Semua angka kesimpulan dihasilkan dari kode, bukan angka ilustrasi.

## Langkah 1 Menyiapkan alat dan konfigurasi
`numpy` mengolah angka, `pandas` mengolah tabel, `matplotlib` membuat grafik, dan `joblib` menyimpan model. Seed 42 membantu pengulangan. Fungsi bersama dipakai untuk menjaga definisi metrik, data, dan preprocessing tetap sama di empat notebook. Bagian pelatihan model ditulis langsung di notebook ini.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
while not (ROOT / 'src/experiment.py').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / 'src/experiment.py').exists(), 'Buka notebook dari folder proyek FoodSave.'
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import json
import numpy as np
import pandas as pd
from IPython.display import display, Image, Markdown, Code
from src.experiment import OUT, ART, load_data, prepare, run_model, evaluate
pd.set_option('display.max_columns', 40)
pd.set_option('display.precision', 4)
get_ipython().run_line_magic('matplotlib', 'inline')

import time
import hashlib
import inspect
import joblib
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from src.experiment import NUM, CAT, TARGET, SEED, WINDOW, FoodPreprocessor, sequence, metrics, predict, plot_curve, save_json
MODEL = 'XGBoost'
CONFIGS = [{'max_depth':3,'reg_lambda':5.0}, {'max_depth':5,'reg_lambda':10.0}]


## Langkah 2 Membaca data dan memastikan urutan waktu
Training memakai tanggal awal, validasi tanggal berikutnya, dan test tanggal terbaru. Tidak ada tanggal yang masuk dua partisi. Duplikat persis tanpa ID dibuang untuk semua model agar tidak muncul salinan observasi yang sama pada train/test. Dataset test Kaggle tanpa label tidak digunakan sebagai holdout penelitian.

In [2]:
df = load_data()
display(df.head(8))
display(df.groupby('split').agg(jumlah=('date','size'), tanggal_awal=('date','min'), tanggal_akhir=('date','max')))
display(json.loads((OUT/'data_audit.json').read_text(encoding='utf-8')))
assert df.loc[df.split=='train','date'].max() < df.loc[df.split=='validation','date'].min()
assert df.loc[df.split=='validation','date'].max() < df.loc[df.split=='test','date'].min()


,ID,date,meals_served,kitchen_staff,temperature_C,humidity_percent,day_of_week,special_event,past_waste_kg,staff_experience,waste_category,food_waste_kg,_row,split
0,397,2022-01-01,202,9,31.9768,66.8266,5,0,45.4455,Intermediate,MEAT,39.9011,341,train
1,427,2022-01-01,202,9,32.3188,66.8266,5,0,45.4455,Intermediate,MEAT,39.9011,364,train
2,201,2022-01-02,448,16,10.0925,56.1303,6,0,29.8900,intermediate,GRAINS,53.2034,171,train
3,983,2022-01-02,448,16,10.1286,56.1303,6,0,29.8900,intermediate,GRAINS,53.2034,849,train
4,162,2022-01-03,370,14,30.4987,64.6350,0,0,41.5516,Beginner,Vegetables,61.3754,141,train
5,764,2022-01-03,370,14,30.7597,64.6350,0,0,41.5516,Beginner,Vegetables,61.3754,656,train
6,50,2022-01-04,206,11,15.7402,74.2841,1,0,22.2819,Beginner,MeAt,32.4411,47,train
7,112,2022-01-04,206,11,15.7156,74.2841,1,0,22.2819,Beginner,MeAt,32.4411,99,train


,jumlah,tanggal_awal,tanggal_akhir
split,,,
test,174,2024-03-11,2024-09-26
train,564,2022-01-01,2023-08-23
validation,173,2023-08-24,2024-03-10


{'source_sha256': 'c2bedb198bf40cc6793921cdfe0573589923996b3347b3361561b114214b9adf',
 'raw_rows': 911,
 'invalid_date_or_target_removed': 0,
 'exact_duplicates_removed_for_all_models': 0,
 'usable_rows': 911,
 'missing_before': {'ID': 0,
  'date': 0,
  'meals_served': 0,
  'kitchen_staff': 0,
  'temperature_C': 0,
  'humidity_percent': 0,
  'day_of_week': 0,
  'special_event': 0,
  'past_waste_kg': 0,
  'staff_experience': 164,
  'waste_category': 0,
  'food_waste_kg': 0},
 'excluded_features': ['ID', 'waste_category', 'past_waste_kg'],
 'sequence_warning': '7 observasi, bukan 7 hari. Tidak ada ID restoran; urutan temporal tidak membuktikan satu seri restoran.',
 'split': {'test': {'n': 174, 'start': '2024-03-11', 'end': '2024-09-26'},
  'train': {'n': 564, 'start': '2022-01-01', 'end': '2023-08-23'},
  'validation': {'n': 173, 'start': '2023-08-24', 'end': '2024-03-10'}}}

## Langkah 3 Memahami input dan mencegah kebocoran informasi
Hanya informasi yang diasumsikan tersedia ketika merencanakan operasional yang digunakan. ID, kategori limbah, dan limbah sebelumnya dikeluarkan; asal/waktu ketersediaannya belum terverifikasi. Hari dihitung dari tanggal. Suhu/kelembapan harus berupa perkiraan jika nilai aktual belum diketahui. Target food_waste_kg tidak pernah menjadi input.

Standard memakai konversi sederhana, imputasi median/modus training, dan kategori apa adanya. Robust menambah pembersihan simbol/spasi, normalisasi kategori, batas IQR training, dan RobustScaler untuk LSTM. Model pohon tidak memerlukan scaling. Target aktual tidak dipotong. Rasio dihitung setelah pembersihan agar tidak membagi dengan nilai nol.

In [3]:
display(Code(inspect.getsource(FoodPreprocessor), language='python'))

class FoodPreprocessor:
    """Standard: konversi sederhana. Robust: normalisasi dan batas IQR training."""
    def __init__(self, method='standard'):
        self.method = method

    def clean(self, df):
        d = df[NUM + CAT].copy()
        for c in NUM:
            s = d[c]
            if self.method == 'robust':
                s = s.astype('string').str.strip().str.replace(',', '', regex=False).str.replace(r'[^0-9.eE+\-]', '', regex=True)
            # Float mendukung median/batas IQR pecahan, termasuk pada pandas 3.
            d[c] = pd.to_numeric(s, errors='coerce').astype('float64').replace([np.inf, -np.inf], np.nan)
        for c in CAT:
            s = d[c].astype('string')
            if self.method == 'robust':
                s = s.str.strip().str.lower().replace({'nan': pd.NA, 'none': pd.NA, '': pd.NA})
            d[c] = s
        date = pd.to_datetime(df['date'])
        d['is_weekend'] = (date.dt.dayofweek >= 5).astype(float)
        d['month_sin'] = np.sin(2 * np.pi * date.dt.month / 12)
        d['month_cos'] = np.cos(2 * np.pi * date.dt.month / 12)
        return d

    def fit(self, df):
        d = self.clean(df)
        self.numerics = d.select_dtypes(include='number').columns.tolist()
        self.medians = d[self.numerics].median().fillna(0)
        self.modes = {c: (d[c].dropna().mode().iloc[0] if d[c].notna().any() else 'unknown') for c in CAT}
        self.bounds = {}
        if self.method == 'robust':
            for c in ['meals_served', 'kitchen_staff', 'temperature_C', 'humidity_percent']:
                q1, q3 = d[c].quantile([.25, .75]); iqr = q3 - q1
                self.bounds[c] = (float(q1 - 1.5 * iqr), float(q3 + 1.5 * iqr))
        native = self.native(df)
        self.columns = pd.get_dummies(native, columns=CAT, dtype=float).columns.tolist()
        self.scaler = (RobustScaler() if self.method == 'robust' else StandardScaler())
        self.scaler.fit(self.encoded(df))
        return self

    def native(self, df):
        d = self.clean(df)
        d[self.numerics] = d[self.numerics].fillna(self.medians)
        for c, bounds in self.bounds.items():
            d[c] = d[c].clip(*bounds)
        for c in CAT:
            d[c] = d[c].fillna(self.modes[c]).astype(str)
        # Rasio dihitung setelah imputasi dan penanganan outlier.
        d['staff_per_portion'] = d.kitchen_staff / d.meals_served.clip(lower=1)
        d['portion_per_staff'] = d.meals_served / d.kitchen_staff.clip(lower=1)
        d['special_event_portion'] = d.special_event * d.meals_served
        return d

    def encoded(self, df):
        return pd.get_dummies(self.native(df), columns=CAT, dtype=float).reindex(columns=self.columns, fill_value=0).astype(float)

    def transform(self, df, scaled=False):
        x = self.encoded(df)
        return self.scaler.transform(x).astype('float32') if scaled else x

## Langkah 4 Menyiapkan fitur dan baris evaluasi yang setara
Statistik preprocessing dipelajari dari training saja, lalu dipakai pada validasi/test. Tujuh observasi pertama diperlukan untuk jendela LSTM. Enam baris awal tidak dijadikan target pada seluruh model agar baris pembanding identik. Kategori CatBoost tetap berupa teks; RF/XGBoost memakai one-hot. LSTM memakai urutan fitur terskala. Tidak ada label test yang digunakan untuk konteks.

In [4]:
eligible = np.arange(len(df)) >= WINDOW-1
idx = {s: np.flatnonzero((df.split==s).to_numpy() & eligible) for s in ['train','validation','test']}
data = {}
for method in ['standard','robust']:
    prep = FoodPreprocessor(method).fit(df[df.split=='train'])
    x = prep.transform(df)
    scaled = prep.transform(df, scaled=True)
    native = prep.native(df)
    target_scaler = StandardScaler().fit(df.loc[idx['train'],TARGET].to_numpy().reshape(-1,1))
    y = {s:df.loc[ix,TARGET].to_numpy() for s,ix in idx.items()}
    if MODEL=='LSTM':
        features = {s:sequence(scaled,ix) for s,ix in idx.items()}
    elif MODEL=='CatBoost':
        features = {s:native.iloc[ix] for s,ix in idx.items()}
    else:
        features = {s:x.iloc[ix] for s,ix in idx.items()}
    assert np.isfinite(x.to_numpy()).all() and np.isfinite(scaled).all()
    assert not set(['ID',TARGET,'waste_category','past_waste_kg']).intersection(x.columns)
    data[method] = {'prep':prep,'ys':target_scaler,'y':y,'features':features,'names':native.columns.tolist() if MODEL=='CatBoost' else x.columns.tolist()}
    print(method, '| jumlah fitur:', x.shape[1], '| bentuk input:', {s:v.shape for s,v in features.items()})
    display(native.head(4))


standard | jumlah fitur: 15 | bentuk input: {'train': (558, 15), 'validation': (173, 15), 'test': (174, 15)}


,meals_served,kitchen_staff,temperature_C,humidity_percent,special_event,staff_experience,is_weekend,month_sin,month_cos,staff_per_portion,portion_per_staff,special_event_portion
0,202.0,9.0,31.9768,66.8266,0.0,Intermediate,1.0,0.5,0.866,0.0446,22.4444,0.0
1,202.0,9.0,32.3188,66.8266,0.0,Intermediate,1.0,0.5,0.866,0.0446,22.4444,0.0
2,448.0,16.0,10.0925,56.1303,0.0,intermediate,1.0,0.5,0.866,0.0357,28.0000,0.0
3,448.0,16.0,10.1286,56.1303,0.0,intermediate,1.0,0.5,0.866,0.0357,28.0000,0.0


robust | jumlah fitur: 14 | bentuk input: {'train': (558, 14), 'validation': (173, 14), 'test': (174, 14)}


,meals_served,kitchen_staff,temperature_C,humidity_percent,special_event,staff_experience,is_weekend,month_sin,month_cos,staff_per_portion,portion_per_staff,special_event_portion
0,202.0,9.0,31.9768,66.8266,0.0,intermediate,1.0,0.5,0.866,0.0446,22.4444,0.0
1,202.0,9.0,32.3188,66.8266,0.0,intermediate,1.0,0.5,0.866,0.0446,22.4444,0.0
2,448.0,16.0,10.0925,56.1303,0.0,intermediate,1.0,0.5,0.866,0.0357,28.0000,0.0
3,448.0,16.0,10.1286,56.1303,0.0,intermediate,1.0,0.5,0.866,0.0357,28.0000,0.0


## Langkah 5 Menulis cara pelatihan model
Fungsi berikut berisi kode model yang benar-benar dilatih, bukan placeholder. Training memperbarui parameter dari data latihan. Validasi memantau kemampuan di periode berikutnya. Early stopping menghentikan jaringan/boosting ketika validasi tidak membaik, lalu memakai keadaan terbaik. RF melihat kestabilan error ketika jumlah pohon bertambah.

Dua konfigurasi diuji per metode, bukan pencarian optimum global. Learning rate adalah besar langkah belajar; kedalaman pohon mengatur kerumitan aturan; regularisasi/minimum daun membatasi model agar tidak terlalu menyesuaikan training.

In [5]:
def latih_kandidat(config, xtr, ytr, xva, yva, target_scaler):
    start = time.perf_counter()
    from xgboost import XGBRegressor
    model = XGBRegressor(n_estimators=500, learning_rate=0.04, objective='reg:squarederror', eval_metric='rmse', early_stopping_rounds=40, random_state=SEED, n_jobs=2, **config)
    model.fit(xtr, ytr, eval_set=[(xtr, ytr), (xva, yva)], verbose=False)
    h = model.evals_result()
    curve = pd.DataFrame({'train_RMSE': h['validation_0']['rmse'], 'validation_RMSE': h['validation_1']['rmse']})
    curve['step'] = np.arange(1, len(curve) + 1)
    return model, curve, time.perf_counter() - start

## Langkah 6 Mencoba kandidat dan menghitung skor validasi
Bagian ini membutuhkan waktu komputasi. Semua kandidat dilatih hanya pada training. Masing-masing dibandingkan menggunakan MAE validasi dalam kg. Data test belum digunakan untuk memilih konfigurasi. Kurva training/validasi dan waktu setiap kandidat dicatat agar prosesnya dapat diaudit.

In [6]:
candidates = {}
tuning_rows = []
for method, d in data.items():
    candidates[method] = []
    for config in CONFIGS:
        model, curve, duration = latih_kandidat(config, d['features']['train'], d['y']['train'], d['features']['validation'], d['y']['validation'], d['ys'])
        prediction = predict(model, d['features']['validation'], MODEL, d['ys'])
        score = metrics(d['y']['validation'], prediction)
        tuning_rows.append({'preprocessing':method, 'config':json.dumps(config), **score, 'train_time_s':duration})
        candidates[method].append({'model':model,'curve':curve,'duration':duration,'config':config,'score':score})
        print(method, config, '| MAE validasi:', round(score['MAE'],4), 'kg', flush=True)
tuning = pd.DataFrame(tuning_rows)
display(tuning.sort_values(['preprocessing','MAE']))


standard {'max_depth': 3, 'reg_lambda': 5.0} | MAE validasi: 10.7636 kg


standard {'max_depth': 5, 'reg_lambda': 10.0} | MAE validasi: 10.7964 kg


robust {'max_depth': 3, 'reg_lambda': 5.0} | MAE validasi: 10.3407 kg


robust

 {'max_depth': 5, 'reg_lambda': 10.0} | MAE validasi: 10.8679 kg


,preprocessing,config,MAE,RMSE,R2,MedAE,Bias,WAPE_percent,train_time_s
2,robust,"{""max_depth"": 3, ""reg_lambda"": 5.0}",10.3407,15.6681,0.7956,7.5389,-1.5766,21.7665,0.0914
3,robust,"{""max_depth"": 5, ""reg_lambda"": 10.0}",10.8679,16.7331,0.7669,8.4416,-1.5080,22.8762,0.1842
0,standard,"{""max_depth"": 3, ""reg_lambda"": 5.0}",10.7636,17.5917,0.7424,7.7571,-1.3114,22.6568,1.2875
1,standard,"{""max_depth"": 5, ""reg_lambda"": 10.0}",10.7964,17.8045,0.7361,7.6699,-2.4791,22.7258,0.1089


## Langkah 7 Mengunci konfigurasi dan membaca kurva belajar
MAE validasi terendah memilih satu kandidat pada Standard dan satu pada Robust. Kurva yang turun berarti error mengecil; kurva tidak harus selalu turun. Error training yang jauh lebih rendah daripada validasi bisa menandakan overfitting atau perubahan distribusi waktu. Sumbu RF adalah jumlah pohon, sumbu boosting adalah tahap, dan sumbu LSTM adalah epoch. RF tidak memakai gradient descent.

In [7]:
selected = {method:min(items,key=lambda item:item['score']['MAE']) for method,items in candidates.items()}
for method,item in selected.items():
    label=f'{MODEL}_{method}'
    item['curve'].to_csv(OUT/f'{label}_curve.csv',index=False)
    tuning[tuning.preprocessing==method].drop(columns='preprocessing').to_csv(OUT/f'{label}_tuning.csv',index=False)
    plot_curve(item['curve'],label)
    print('Terpilih:',method,item['config'])
    display(Image(filename=str(OUT/f'{label}_curve.png')))


Terpilih: standard {'max_depth': 3, 'reg_lambda': 5.0}


Terpilih: robust {'max_depth': 3, 'reg_lambda': 5.0}


## Langkah 8 Mengukur seluruh metrik training validasi dan test
MAE: kesalahan absolut rata-rata dalam kg. RMSE: lebih sensitif terhadap error besar. R²: kemampuan menjelaskan variasi; dapat negatif. MedAE: median error absolut. Bias: prediksi dikurangi aktual; positif berarti terlalu tinggi. WAPE: total error absolut dibanding total limbah, bukan akurasi klasifikasi.

Waktu prediksi berikut diukur untuk satu batch tanpa preprocessing dan pemuatan model. Milidetik per baris adalah waktu batch dibagi jumlah baris, bukan benchmark permintaan tunggal. Waktu training mencakup pelatihan dan pemantauan kurva kandidat terpilih, termasuk impor awal library/pengukuran gradien; total tuning mencakup kedua kandidat. Model tidak dilatih ulang pada validasi.

In [8]:
result_rows = []
protocol = hashlib.sha256((ROOT/'src/experiment.py').read_bytes()+(ROOT/'dataset/raw/train.csv').read_bytes()).hexdigest()
for method,item in selected.items():
    d=data[method];label=f'{MODEL}_{method}'
    row={'model':MODEL,'preprocessing':method,'config':json.dumps(item['config']),'protocol_sha256':protocol,
         'train_time_s':item['duration'],'tuning_total_time_s':float(tuning.loc[tuning.preprocessing==method,'train_time_s'].sum()),
         **{f'n_{s}':len(v) for s,v in d['y'].items()}}
    for split in ['train','validation','test']:
        start=time.perf_counter()
        prediction=predict(item['model'],d['features'][split],MODEL,d['ys'])
        elapsed=time.perf_counter()-start
        row.update({f'{split}_{k}':v for k,v in metrics(d['y'][split],prediction).items()})
        row[f'{split}_prediction_time_s']=elapsed
        row[f'{split}_ms_per_row']=elapsed/len(prediction)*1000
        pd.DataFrame({'row':df.loc[idx[split],'_row'],'date':df.loc[idx[split],'date'],'actual':d['y'][split],
                      'prediction':prediction,'residual':d['y'][split]-prediction}).to_csv(OUT/f'{label}_{split}_predictions.csv',index=False)
    save_json(OUT/f'{label}_metrics.json',row)
    result_rows.append(row)
hasil=pd.DataFrame(result_rows)
for split in ['train','validation','test']:
    display(Markdown(f'### Metrik {split}'))
    display(hasil[['model','preprocessing']+[f'{split}_{m}' for m in ['MAE','RMSE','R2','MedAE','Bias','WAPE_percent']]])
display(hasil[['model','preprocessing','train_time_s','tuning_total_time_s','test_prediction_time_s','test_ms_per_row']])


### Metrik train

,model,preprocessing,train_MAE,train_RMSE,train_R2,train_MedAE,train_Bias,train_WAPE_percent
0,XGBoost,standard,6.7639,8.3612,0.8978,6.0093,-0.0790,15.2951
1,XGBoost,robust,7.1426,8.9335,0.8833,6.2632,-0.1919,16.1516


### Metrik validation

,model,preprocessing,validation_MAE,validation_RMSE,validation_R2,validation_MedAE,validation_Bias,validation_WAPE_percent
0,XGBoost,standard,10.7636,17.5917,0.7424,7.7571,-1.3114,22.6568
1,XGBoost,robust,10.3407,15.6681,0.7956,7.5389,-1.5766,21.7665


### Metrik test

,model,preprocessing,test_MAE,test_RMSE,test_R2,test_MedAE,test_Bias,test_WAPE_percent
0,XGBoost,standard,9.2553,14.7333,0.6806,7.0804,-0.5681,21.0571
1,XGBoost,robust,9.1857,14.3421,0.6973,7.0124,-0.7461,20.8987


,model,preprocessing,train_time_s,tuning_total_time_s,test_prediction_time_s,test_ms_per_row
0,XGBoost,standard,1.2875,1.3964,0.0033,0.0189
1,XGBoost,robust,0.0914,0.2756,0.0025,0.0145


## Langkah 9 Melihat prediksi dan kasus yang paling sulit
Garis diagonal menandai prediksi sempurna. Residual positif berarti model meremehkan limbah. Histogram yang lebar berarti variasi error besar. Tabel error terbesar membantu mencari kondisi operasional yang sulit, tetapi tidak digunakan lagi untuk menyetel model; jika dilakukan, dibutuhkan test baru.

In [9]:
for method in ['standard','robust']:
    pred=pd.read_csv(OUT/f'{MODEL}_{method}_test_predictions.csv')
    fig,axes=plt.subplots(1,3,figsize=(15,4))
    axes[0].scatter(pred.actual,pred.prediction,alpha=.6)
    limits=[min(pred.actual.min(),pred.prediction.min()),max(pred.actual.max(),pred.prediction.max())]
    axes[0].plot(limits,limits,'r--');axes[0].set(xlabel='Aktual kg',ylabel='Prediksi kg',title=f'{MODEL} {method}')
    axes[1].scatter(pred.prediction,pred.residual,alpha=.6);axes[1].axhline(0,color='red');axes[1].set(xlabel='Prediksi kg',ylabel='Aktual - prediksi kg')
    axes[2].hist(pred.residual,bins=20);axes[2].axvline(0,color='red');axes[2].set_title('Sebaran error kg')
    fig.tight_layout();fig.savefig(OUT/f'{MODEL}_{method}_diagnostics.png',dpi=140);display(fig);plt.close(fig)
    context=df[['_row','meals_served','kitchen_staff','special_event','staff_experience']]
    difficult=pred.merge(context,left_on='row',right_on='_row',how='left')
    display(difficult.assign(error_absolut=difficult.residual.abs()).nlargest(10,'error_absolut'))


,row,date,actual,prediction,residual,_row,meals_served,kitchen_staff,special_event,staff_experience,error_absolut
28,265,2024-04-12,175.5071,71.8610,103.6461,265,2440,13,0,EXPERT,103.6461
19,380,2024-03-31,168.8688,71.3084,97.5604,380,4100,18,0,intermediate,97.5604
126,451,2024-08-05,167.7235,203.6869,-35.9634,451,3170,14,0,Intermediate,35.9634
155,631,2024-09-05,21.5655,46.3128,-24.7473,631,177,5,1,Intermediate,24.7473
58,131,2024-05-16,21.5092,43.1102,-21.6010,131,328,12,0,EXPERT,21.6010
66,730,2024-05-26,65.4262,43.8737,21.5526,730,405,15,0,Beginner,21.5526
15,276,2024-03-27,52.8336,32.3148,20.5188,276,197,11,0,Beginner,20.5188
120,531,2024-07-25,59.6054,39.6358,19.9696,531,416,9,0,Beginner,19.9696
87,489,2024-06-17,64.1462,45.2055,18.9408,489,439,15,0,Intermediate,18.9408
18,696,2024-03-30,41.2591,60.1494,-18.8904,696,484,5,0,EXPERT,18.8904


,row,date,actual,prediction,residual,_row,meals_served,kitchen_staff,special_event,staff_experience,error_absolut
19,380,2024-03-31,168.8688,71.3729,97.4958,380,4100,18,0,intermediate,97.4958
28,265,2024-04-12,175.5071,79.5830,95.9242,265,2440,13,0,EXPERT,95.9242
126,451,2024-08-05,167.7235,196.5785,-28.8550,451,3170,14,0,Intermediate,28.8550
155,631,2024-09-05,21.5655,46.5519,-24.9864,631,177,5,1,Intermediate,24.9864
58,131,2024-05-16,21.5092,43.4629,-21.9537,131,328,12,0,EXPERT,21.9537
66,730,2024-05-26,65.4262,44.3281,21.0981,730,405,15,0,Beginner,21.0981
157,518,2024-09-07,67.2923,47.1970,20.0953,518,485,15,0,Intermediate,20.0953
15,276,2024-03-27,52.8336,32.7961,20.0375,276,197,11,0,Beginner,20.0375
120,531,2024-07-25,59.6054,40.4338,19.1716,531,416,9,0,Beginner,19.1716
87,489,2024-06-17,64.1462,45.2094,18.9368,489,439,15,0,Intermediate,18.9368


## Langkah 10 Menilai fitur yang membantu prediksi
Permutation importance mengacak satu fitur pada validasi sebanyak tiga kali. Kenaikan MAE positif berarti fitur tersebut membantu model. Nilai negatif dapat terjadi akibat noise atau model yang terlalu mengandalkan fitur tersebut. Ini bukan bukti kausal. Rasio dan porsi saling berkaitan sehingga importance dapat terbagi. Pada LSTM, satu jalur fitur sepanjang jendela diacak sebagai satu blok; hasil bersifat diagnostik.

In [10]:
for method,item in selected.items():
    d=data[method];rng=np.random.default_rng(SEED);importance=[]
    for j,feature in enumerate(d['names']):
        differences=[]
        for repeat in range(3):
            shuffled=d['features']['validation'].copy();order=rng.permutation(len(shuffled))
            if MODEL=='LSTM':shuffled[:,:,j]=shuffled[order,:,j]
            else:shuffled.iloc[:,j]=shuffled.iloc[order,j].to_numpy()
            score=metrics(d['y']['validation'],predict(item['model'],shuffled,MODEL,d['ys']))['MAE']
            differences.append(score-item['score']['MAE'])
        importance.append({'feature':feature,'MAE_increase':np.mean(differences),'std':np.std(differences)})
    imp=pd.DataFrame(importance).sort_values('MAE_increase',ascending=False)
    imp.to_csv(OUT/f'{MODEL}_{method}_importance.csv',index=False)
    display(imp)
    top=imp.head(12).sort_values('MAE_increase')
    fig,ax=plt.subplots(figsize=(9,4));ax.barh(top.feature,top.MAE_increase,xerr=top['std'])
    ax.set(xlabel='Kenaikan MAE validasi kg',title=f'Pengaruh fitur {MODEL} {method}')
    fig.tight_layout();display(fig);plt.close(fig)


,feature,MAE_increase,std
0,meals_served,7.8588,1.0978
6,month_sin,0.7414,0.6445
10,special_event_portion,0.2821,0.0694
1,kitchen_staff,0.1769,0.0164
4,special_event,0.0961,0.0172
8,staff_per_portion,0.0416,0.1255
5,is_weekend,0.0000,0.0000
11,staff_experience_Beginner,0.0000,0.0000
7,month_cos,-0.0074,0.0085
14,staff_experience_intermediate,-0.0120,0.0059


,feature,MAE_increase,std
0,meals_served,9.2920,0.7177
6,month_sin,0.4233,0.5363
10,special_event_portion,0.3010,0.0722
1,kitchen_staff,0.2197,0.0100
4,special_event,0.0779,0.0266
5,is_weekend,0.0000,0.0000
11,staff_experience_beginner,0.0000,0.0000
13,staff_experience_intermediate,-0.0009,0.0007
7,month_cos,-0.0083,0.0078
8,staff_per_portion,-0.0185,0.0119


## Langkah 11 Menyimpan model bersama preprocessing
Model saja tidak cukup. Kategori, median, batas outlier, skala, dan urutan fitur harus sama saat digunakan kembali. Model lama milik tim tidak ditimpa: artefak baru memakai nama model/metode tersendiri di folder models/best_model yang sudah ada. Integrasi ke API merupakan langkah terpisah.

In [11]:
for method,item in selected.items():
    label=f'{MODEL}_{method}'
    if MODEL=='LSTM':item['model'].save(ART/f'{label}.keras')
    else:joblib.dump(item['model'],ART/f'{label}.joblib')
    joblib.dump({'preprocessor':data[method]['prep'],'target_scaler':data[method]['ys'],'window':WINDOW,'name':MODEL},ART/f'{label}_preprocessing.joblib')
    print('Tersimpan:',label)


Tersimpan: XGBoost_standard
Tersimpan: XGBoost_robust


## Langkah 12 Menulis insight dari angka yang benar-benar diperoleh
Kesimpulan berikut berubah mengikuti hasil. MAE test tidak dipakai memilih pemenang. Robust bisa lebih buruk jika clipping menghilangkan nilai ekstrem yang sebenarnya bermakna. Satu split/seed tidak membuktikan keunggulan universal. Setelah notebook model selesai, buka notebook 08 untuk perbandingan delapan eksperimen.

In [12]:
a=hasil.set_index('preprocessing').loc['standard'];b=hasil.set_index('preprocessing').loc['robust']
delta=a.validation_MAE-b.validation_MAE
display(Markdown(f'**Dampak Robust pada validasi:** MAE berubah dari {a.validation_MAE:.3f} menjadi {b.validation_MAE:.3f} kg. Selisih Standard − Robust = {delta:.3f} kg. Positif berarti Robust lebih baik.'))
baseline_value=float(df.loc[idx['train'],TARGET].median())
baseline_mae=metrics(df.loc[idx['test'],TARGET].to_numpy(),np.full(len(idx['test']),baseline_value))['MAE']
for _,r in hasil.iterrows():
    improvement=(baseline_mae-r.test_MAE)/baseline_mae*100
    display(Markdown(f"**{r.preprocessing}:** training MAE {r.train_MAE:.3f} kg, validasi {r.validation_MAE:.3f} kg, test {r.test_MAE:.3f} kg. Selisih validasi − training {r.validation_MAE-r.train_MAE:.3f} kg. R² test {r.test_R2:.3f}. Pengurangan error dibanding tebakan median training pada test {improvement:.1f}%."))
display(Markdown('Gunakan tabel error terbesar dan grafik residual untuk menjelaskan kelemahan model. Jangan mengubah parameter berdasarkan kasus test lalu tetap menyebut hasil test tersebut evaluasi independen.'))


**Dampak Robust pada validasi:** MAE berubah dari 10.764 menjadi 10.341 kg. Selisih Standard − Robust = 0.423 kg. Positif berarti Robust lebih baik.

**standard:** training MAE 6.764 kg, validasi 10.764 kg, test 9.255 kg. Selisih validasi − training 4.000 kg. R² test 0.681. Pengurangan error dibanding tebakan median training pada test 28.4%.

**robust:** training MAE 7.143 kg, validasi 10.341 kg, test 9.186 kg. Selisih validasi − training 3.198 kg. R² test 0.697. Pengurangan error dibanding tebakan median training pada test 29.0%.

Gunakan tabel error terbesar dan grafik residual untuk menjelaskan kelemahan model. Jangan mengubah parameter berdasarkan kasus test lalu tetap menyebut hasil test tersebut evaluasi independen.

## Langkah 13 Membuka kembali artefak untuk melanjutkan pekerjaan
Contoh ini memeriksa model tersimpan pada lima observasi test yang sama, bukan pengujian baru. Jika prediksinya sama, pasangan model/preprocessing dapat dibuka kembali tanpa training. Saat dipakai pada input baru, urutan kolom dan asumsi ketersediaan fitur harus tetap sama. Model LSTM membutuhkan konteks, bukan hanya satu baris.

In [13]:
for method in ['standard','robust']:
    label=f'{MODEL}_{method}'
    bundle=joblib.load(ART/f'{label}_preprocessing.joblib')
    if MODEL=='LSTM':
        import tensorflow as tf
        loaded=tf.keras.models.load_model(ART/f'{label}.keras',compile=False)
        encoded=bundle['preprocessor'].transform(df,scaled=True)
        sample=sequence(encoded,idx['test'][:5])
    else:
        loaded=joblib.load(ART/f'{label}.joblib')
        transformed=bundle['preprocessor'].native(df) if MODEL=='CatBoost' else bundle['preprocessor'].transform(df)
        sample=transformed.iloc[idx['test'][:5]]
    restored=predict(loaded,sample,MODEL,bundle['target_scaler'])
    saved=pd.read_csv(OUT/f'{label}_test_predictions.csv').prediction.iloc[:5].to_numpy()
    np.testing.assert_allclose(restored,saved,rtol=1e-5,atol=1e-5)
    display(pd.DataFrame({'prediksi_tersimpan':saved,'prediksi_setelah_dibuka':restored}))
    print(method,': LULUS, model dapat dibuka kembali')


,prediksi_tersimpan,prediksi_setelah_dibuka
0,40.1367,40.1367
1,45.1040,45.1040
2,30.4124,30.4124
3,35.6957,35.6958
4,44.7118,44.7118


standard : LULUS, model dapat dibuka kembali


,prediksi_tersimpan,prediksi_setelah_dibuka
0,40.7870,40.7870
1,45.0436,45.0436
2,31.5126,31.5126
3,35.6608,35.6608
4,45.3160,45.3160


robust : LULUS, model dapat dibuka kembali
